In [1]:
import pandas as pd

In [5]:
import sqlite3

In [6]:
df1 = pd.read_excel("/content/order_region_a.xlsx")

In [7]:
df1.head()

,OrderId,OrderItemId,QuantityOrdered,ItemPrice,PromotionDiscount,batch_id
0,171-0001135-1657958,11168926687715,1,949.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10""}",359
1,171-0001497-9165123,19760298917699,1,699.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10.1""}",1135
2,171-0002127-1363507,5949764099083,1,399.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10""}",297
3,171-0002370-0601169,57571868836379,1,499.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10.1""}",114
4,171-0004526-2028348,33851287891403,1,1699.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10""}",764


In [8]:
df2 = pd.read_excel("/content/order_region_b.xlsx")

In [9]:
df2.head()

,OrderId,OrderItemId,QuantityOrdered,ItemPrice,PromotionDiscount,batch_id
0,171-0001135-1657958,11168926687715,1,949.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10""}",359
1,171-0001497-9165123,19760298917699,1,699.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10.1""}",1135
2,171-0002127-1363507,5949764099083,1,399.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10""}",297
3,171-0002370-0601169,57571868836379,1,499.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10.1""}",114
4,171-0004526-2028348,33851287891403,1,1699.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10""}",764


In [11]:
df1["region"] = "A"
df2["region"] = "B"

In [16]:
df = pd.concat([df1, df2], ignore_index = True)

In [17]:
df.head()

,OrderId,OrderItemId,QuantityOrdered,ItemPrice,PromotionDiscount,batch_id,region
0,171-0001135-1657958,11168926687715,1,949.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10""}",359,A
1,171-0001497-9165123,19760298917699,1,699.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10.1""}",1135,A
2,171-0002127-1363507,5949764099083,1,399.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10""}",297,A
3,171-0002370-0601169,57571868836379,1,499.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10.1""}",114,A
4,171-0004526-2028348,33851287891403,1,1699.0,"{ ""CurrencyCode"": ""INR"", ""Amount"": ""10""}",764,A


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 309 entries, 0 to 43655
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   OrderId            309 non-null    object 
 1   OrderItemId        309 non-null    int64  
 2   QuantityOrdered    309 non-null    int64  
 3   ItemPrice          309 non-null    float64
 4   PromotionDiscount  309 non-null    int64  
 5   batch_id           309 non-null    int64  
 6   region             309 non-null    object 
 7   total_sales        309 non-null    float64
 8   net_sales          309 non-null    float64
dtypes: float64(3), int64(4), object(2)
memory usage: 24.1+ KB


In [25]:
df["QuantityOrdered"] = pd.to_numeric(df["QuantityOrdered"], errors='coerce').fillna(0)
df["ItemPrice"] = pd.to_numeric(df["ItemPrice"], errors='coerce').fillna(0)
df["PromotionDiscount"] = pd.to_numeric(df["PromotionDiscount"], errors='coerce').fillna(0)

In [26]:
df["total_sales"] = df["QuantityOrdered"] * df["ItemPrice"]

In [29]:
df["net_sales"] = df["total_sales"] - df["PromotionDiscount"]

In [30]:
df=df.drop_duplicates(subset=["total_sales"])

In [35]:
df= df[df["net_sales"]> 0]

In [40]:
conn = sqlite3.connect("sales.db")

In [42]:
df.to_sql("sales_data", conn, if_exists="replace", index=False)

309

In [51]:
a = pd.read_sql("SELECT COUNT(*) FROM sales_data", conn)
print(a)

   COUNT(*)
0       309


In [52]:
b = pd.read_sql("SELECT region, SUM(net_sales) FROM sales_data GROUP BY region", conn)
print(b)

  region  SUM(net_sales)
0      A      2089153.72


In [55]:
from google.colab import files

In [56]:
files.download("sales.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [59]:
df3 = pd.read_sql("SELECT * FROM sales_data", conn)
display(df3.head())

,OrderId,OrderItemId,QuantityOrdered,ItemPrice,PromotionDiscount,batch_id,region,total_sales,net_sales
0,171-0001135-1657958,11168926687715,1,949.0,0,359,A,949.0,949.0
1,171-0001497-9165123,19760298917699,1,699.0,0,1135,A,699.0,699.0
2,171-0002127-1363507,5949764099083,1,399.0,0,297,A,399.0,399.0
3,171-0002370-0601169,57571868836379,1,499.0,0,114,A,499.0,499.0
4,171-0004526-2028348,33851287891403,1,1699.0,0,764,A,1699.0,1699.0
